In [6]:
# Final tool dependencies
#requests   → API communication
#rasterio   → GeoTIFF validation later
#shapely    → AOI / geometry validation later
#pyproj     → CRS / coordinate operations later
#pydantic   → structured request/response models

!pip -q install requests rasterio shapely pyproj pydantic

In [7]:
# Core imports used throughout the satellite-data tool
import os
import time
import requests
import re
import rasterio

from pathlib import Path
from datetime import datetime, timezone
from dataclasses import dataclass

from dotenv import load_dotenv

load_dotenv()

True

In [8]:
# Central configuration for Sentinel Hub APIs

TOKEN_URL = (
    "https://services.sentinel-hub.com/"
    "auth/realms/main/protocol/openid-connect/token"
)

CATALOG_URL = "https://services.sentinel-hub.com/catalog/v1/search"

PROCESS_URL = "https://services.sentinel-hub.com/process/v1"

OUTPUT_DIR = Path("sih_satellite_data")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")
print("Token URL:", TOKEN_URL)
print("Catalog URL:", CATALOG_URL)
print("Process URL:", PROCESS_URL)
print("Output directory:", OUTPUT_DIR.resolve())

Configuration loaded.
Token URL: https://services.sentinel-hub.com/auth/realms/main/protocol/openid-connect/token
Catalog URL: https://services.sentinel-hub.com/catalog/v1/search
Process URL: https://services.sentinel-hub.com/process/v1
Output directory: /Users/dhruvpandey/Downloads/dhruv coding/satquery/Tools/Tool_1_fetch_satellite_imagery/sih_satellite_data


In [9]:
# Load Sentinel Hub OAuth credentials securely from environment variables (.env)

CLIENT_ID = os.environ.get("SENTINEL_CLIENT_ID")
CLIENT_SECRET = os.environ.get("SENTINEL_CLIENT_SECRET")

if not CLIENT_ID:
    raise RuntimeError(
        "SENTINEL_CLIENT_ID was not found. Set it in a .env file "
        "or export it as an environment variable."
    )

if not CLIENT_SECRET:
    raise RuntimeError(
        "SENTINEL_CLIENT_SECRET was not found. Set it in a .env file "
        "or export it as an environment variable."
    )

print("Sentinel Hub credentials loaded.")
print("Client ID present:", bool(CLIENT_ID))
print("Client Secret present:", bool(CLIENT_SECRET))

Sentinel Hub credentials loaded.
Client ID present: True
Client Secret present: True


In [10]:
# Store tool-wide configuration in one object

@dataclass(frozen=True)
class ToolConfig:
    client_id: str
    client_secret: str
    token_url: str
    catalog_url: str
    process_url: str
    output_dir: Path


CONFIG = ToolConfig(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    token_url=TOKEN_URL,
    catalog_url=CATALOG_URL,
    process_url=PROCESS_URL,
    output_dir=OUTPUT_DIR,
)

print("Tool configuration ready.")
print("Catalog URL:", CONFIG.catalog_url)
print("Process URL:", CONFIG.process_url)
print("Output directory:", CONFIG.output_dir)

Tool configuration ready.
Catalog URL: https://services.sentinel-hub.com/catalog/v1/search
Process URL: https://services.sentinel-hub.com/process/v1
Output directory: sih_satellite_data


In [11]:
# Manage Sentinel Hub OAuth authentication and reuse valid access tokens

class SentinelHubAuth:
    def __init__(self, config: ToolConfig):
        self.config = config
        self._access_token = None
        self._expires_at = 0.0

    def get_token(self) -> str:
        """Return a valid access token, refreshing it when necessary."""

        current_time = time.time()

        # Reuse the existing token if it is still valid.
        if (
            self._access_token is not None
            and current_time < self._expires_at
        ):
            return self._access_token

        response = requests.post(
            self.config.token_url,
            headers={
                "Content-Type": "application/x-www-form-urlencoded",
            },
            data={
                "grant_type": "client_credentials",
                "client_id": self.config.client_id,
                "client_secret": self.config.client_secret,
            },
            timeout=30,
        )

        # Handle authentication failure with useful information.
        if not response.ok:
            try:
                error_data = response.json()
            except ValueError:
                error_data = response.text

            raise RuntimeError(
                f"Sentinel Hub authentication failed "
                f"(HTTP {response.status_code}): {error_data}"
            )

        token_data = response.json()

        access_token = token_data.get("access_token")
        expires_in = token_data.get("expires_in")

        if not access_token:
            raise RuntimeError(
                "Authentication succeeded but no access token was returned."
            )

        if not expires_in:
            raise RuntimeError(
                "Authentication response did not contain token expiry information."
            )

        # Cache the token with a 60-second safety margin.
        self._access_token = access_token
        self._expires_at = (
            current_time + float(expires_in) - 60
        )

        return self._access_token

In [12]:
auth = SentinelHubAuth(CONFIG)

print("Authentication manager initialized.")

Authentication manager initialized.


In [13]:
# Test Sentinel Hub authentication

try:
    token = auth.get_token()

    print("Authentication successful.")
    print("Access token received:", bool(token))

except RuntimeError as e:
    print("Authentication failed.")
    print(e)

except requests.RequestException as e:
    print("Network/API request error.")
    print(e)

except Exception as e:
    print("Unexpected authentication error.")
    print(e)

Authentication successful.
Access token received: True


In [14]:
try:
    token_1 = auth.get_token()
    token_2 = auth.get_token()

    print("Token 1 available:", bool(token_1))
    print("Token 2 available:", bool(token_2))
    print("Same cached token reused:", token_1 == token_2)

except Exception as e:
    print("Authentication error:", e)

Token 1 available: True
Token 2 available: True
Same cached token reused: True


In [15]:
def get_auth_headers(auth: SentinelHubAuth) -> dict:
    """Build authorization headers for Sentinel Hub API requests."""

    token = auth.get_token()

    return {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    }

In [16]:
try:
    headers = get_auth_headers(auth)

    print("Authorization headers prepared.")
    print("Authorization header present:", "Authorization" in headers)

except Exception as e:
    print("Header preparation error:", e)

Authorization headers prepared.
Authorization header present: True


In [17]:
# Imports for request validation and structured tool inputs

from typing import Literal, Optional

from pydantic import BaseModel, Field, field_validator, model_validator

In [18]:
# Define and validate the input structure for satellite imagery requests

class SatelliteRequest(BaseModel):
    bbox: list[float] = Field(
        ...,
        description="Bounding box in [min_lon, min_lat, max_lon, max_lat] order."
    )

    start_date: str
    end_date: str

    modality: Literal["optical", "multispectral"] = "optical"

    bands: Optional[list[str]] = None

    max_cloud_cover: float = Field(
        default=30.0,
        ge=0.0,
        le=100.0
    )

    width: int = Field(
        default=512,
        ge=1,
        le=4096
    )

    height: int = Field(
        default=512,
        ge=1,
        le=4096
    )

    crs: str = "EPSG:4326"

    output_format: Literal["GeoTIFF"] = "GeoTIFF"

    @field_validator("bbox")
    @classmethod
    def validate_bbox(cls, value):
        if len(value) != 4:
            raise ValueError(
                "bbox must contain exactly 4 values: "
                "[min_lon, min_lat, max_lon, max_lat]."
            )

        min_lon, min_lat, max_lon, max_lat = value

        if not -180 <= min_lon <= 180:
            raise ValueError("bbox min_lon must be between -180 and 180.")

        if not -180 <= max_lon <= 180:
            raise ValueError("bbox max_lon must be between -180 and 180.")

        if not -90 <= min_lat <= 90:
            raise ValueError("bbox min_lat must be between -90 and 90.")

        if not -90 <= max_lat <= 90:
            raise ValueError("bbox max_lat must be between -90 and 90.")

        if min_lon >= max_lon:
            raise ValueError("bbox min_lon must be smaller than max_lon.")

        if min_lat >= max_lat:
            raise ValueError("bbox min_lat must be smaller than max_lat.")

        return value

    @field_validator("start_date", "end_date")
    @classmethod
    def validate_date_format(cls, value):
        from datetime import date

        try:
            date.fromisoformat(value)
        except ValueError:
            raise ValueError(
                f"Invalid date '{value}'. Use YYYY-MM-DD format."
            )

        return value

    @model_validator(mode="after")
    def validate_date_range(self):
        from datetime import date

        start = date.fromisoformat(self.start_date)
        end = date.fromisoformat(self.end_date)

        if start > end:
            raise ValueError(
                "start_date cannot be later than end_date."
            )

        return self

    @model_validator(mode="after")
    def validate_band_configuration(self):

        if self.modality == "optical":
            if self.bands is not None:
                raise ValueError(
                    "bands must be omitted when modality='optical'. "
                    "Optical mode always returns B04/B03/B02 RGB."
                )

        if self.modality == "multispectral":
            if not self.bands:
                raise ValueError(
                    "bands must be provided when "
                    "modality='multispectral'."
                )

        return self

In [19]:
# Test validation with an invalid bounding box

try:
    request = SatelliteRequest(
        bbox=[200, 28.50, 77.30, 28.70],
        start_date="2025-01-01",
        end_date="2025-01-31"
    )

    print("Request validation successful.")

except Exception as e:
    print("Request validation failed.")
    print(e)

Request validation failed.
1 validation error for SatelliteRequest
bbox
  Value error, bbox min_lon must be between -180 and 180. [type=value_error, input_value=[200, 28.5, 77.3, 28.7], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


In [20]:
# Test validation with an invalid bounding box

try:
    request = SatelliteRequest(
        bbox=[200, 28.50, 77.30, 28.70],
        start_date="2025-01-01",
        end_date="2025-01-31"
    )

    print("Request validation successful.")

except Exception as e:
    print("Request validation failed.")
    print(e)

Request validation failed.
1 validation error for SatelliteRequest
bbox
  Value error, bbox min_lon must be between -180 and 180. [type=value_error, input_value=[200, 28.5, 77.3, 28.7], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


In [21]:
# Provide shared HTTP handling, including timeouts and transient-error retries

class SentinelHubHTTPClient:
    def __init__(
        self,
        auth: SentinelHubAuth,
        max_retries: int = 3,
        timeout: int = 60,
    ):
        self.auth = auth
        self.max_retries = max_retries
        self.timeout = timeout

        self.retryable_status_codes = {
            429,
            500,
            502,
            503,
            504,
        }

    def _get_headers(
        self,
        content_type: str = "application/json"
    ) -> dict:
        token = self.auth.get_token()

        return {
            "Authorization": f"Bearer {token}",
            "Content-Type": content_type,
        }

    def request(
        self,
        method: str,
        url: str,
        *,
        content_type: str = "application/json",
        **kwargs,
    ) -> requests.Response:

        for attempt in range(self.max_retries + 1):

            try:
                response = requests.request(
                    method=method,
                    url=url,
                    headers=self._get_headers(content_type),
                    timeout=self.timeout,
                    **kwargs,
                )

            except requests.RequestException:
                if attempt >= self.max_retries:
                    raise

                self._sleep_before_retry(attempt)
                continue

            if response.status_code not in self.retryable_status_codes:
                return response

            if attempt >= self.max_retries:
                return response

            retry_after = self._get_retry_after(response)

            if retry_after is not None:
                sleep_time = retry_after
            else:
                sleep_time = self._calculate_backoff(attempt)

            print(
                f"Transient HTTP {response.status_code}. "
                f"Retrying in {sleep_time:.1f} seconds..."
            )

            time.sleep(sleep_time)

        raise RuntimeError("HTTP request failed unexpectedly.")

    def _calculate_backoff(self, attempt: int) -> float:
        return min(2 ** attempt, 30)

    def _sleep_before_retry(self, attempt: int):
        sleep_time = self._calculate_backoff(attempt)

        print(
            f"Network request failed. "
            f"Retrying in {sleep_time:.1f} seconds..."
        )

        time.sleep(sleep_time)

    def _get_retry_after(
        self,
        response: requests.Response
    ) -> float | None:

        retry_after = response.headers.get("Retry-After")

        if not retry_after:
            return None

        try:
            return max(float(retry_after), 0.0)

        except ValueError:
            return None

In [22]:
# Create the shared HTTP client used by all Sentinel Hub API services

http_client = SentinelHubHTTPClient(
    auth=auth,
    max_retries=3,
    timeout=60,
)

print("HTTP client initialized.")

HTTP client initialized.


In [23]:
# Verify that the shared HTTP client can obtain authorization headers

try:
    headers = http_client._get_headers()

    print("HTTP client authentication ready.")
    print("Authorization header present:", "Authorization" in headers)

except Exception as e:
    print("HTTP client initialization error.")
    print(e)

HTTP client authentication ready.
Authorization header present: True


In [24]:
# Search the Sentinel-2 L2A catalog for observations matching the request.
# Returns raw catalog features for further scene processing.

def search_sentinel2_catalog(
    request: SatelliteRequest,
    max_results: int = 20,
) -> list[dict]:

    # Build the time interval required by the Catalog API.
    datetime_range = (
        f"{request.start_date}T00:00:00Z/"
        f"{request.end_date}T23:59:59Z"
    )

    # Build the initial catalog query.
    payload = {
        "bbox": request.bbox,
        "datetime": datetime_range,
        "collections": ["sentinel-2-l2a"],
        "limit": min(10, max_results),
    }

    # Apply cloud filtering when requested.
    if request.max_cloud_cover < 100:
        payload["filter"] = (
            f"eo:cloud_cover <= {request.max_cloud_cover}"
        )

    all_features = []

    while len(all_features) < max_results:

        response = http_client.request(
            "POST",
            CONFIG.catalog_url,
            json=payload,
        )

        if not response.ok:
            try:
                error_data = response.json()
            except ValueError:
                error_data = response.text

            raise RuntimeError(
                f"Catalog search failed "
                f"(HTTP {response.status_code}): {error_data}"
            )

        data = response.json()

        features = data.get("features", [])

        if not isinstance(features, list):
            raise RuntimeError(
                "Catalog response contains an invalid 'features' field."
            )

        all_features.extend(features)

        # Check whether another page exists.
        context = data.get("context", {})
        next_value = context.get("next")

        if not next_value or not features:
            break

        payload["next"] = next_value

        # Prevent accidental over-fetching.
        remaining = max_results - len(all_features)

        if remaining <= 0:
            break

        payload["limit"] = min(10, remaining)

    return all_features[:max_results]

In [25]:
import warnings
warnings.filterwarnings("ignore")

# Create a catalog-search request for an example AOI and time period.

try:
    search_request = SatelliteRequest(
        bbox=[77.10, 28.50, 77.30, 28.70],
        start_date="2025-01-01",
        end_date="2025-01-31",
        modality="optical",
        # bands=["B02", "B03", "B04"],  # Removed as it conflicts with optical modality
        max_cloud_cover=30,
        width=512,
        height=512,
        crs="EPSG:4326"
    )

    scenes = search_sentinel2_catalog(
        search_request,
        max_results=20,
    )

    print("Catalog search successful.")
    print("Scenes found:", len(scenes))

except RuntimeError as e:
    print("Catalog search error.")
    print(e)

except Exception as e:
    print("Unexpected error.")
    print(e)

Catalog search successful.
Scenes found: 5


In [26]:
# Inspect the structure of the first catalog result.

if scenes:
    print("First scene keys:")
    print(scenes[0].keys())

    print("\nScene ID:")
    print(scenes[0].get("id"))

    print("\nProperties:")
    print(scenes[0].get("properties", {}))
else:
    print("No Sentinel-2 scenes were found.")

First scene keys:
dict_keys(['stac_version', 'stac_extensions', 'id', 'type', 'geometry', 'bbox', 'properties', 'assets', 'collection', 'links'])

Scene ID:
S2C_MSIL2A_20250128T053131_N0511_R105_T43RFM_20250128T084454

Properties:
{'datetime': '2025-01-28T05:41:34Z', 'platform': 'sentinel-2c', 'instruments': ['msi'], 'constellation': 'sentinel-2', 'gsd': 10, 'eo:cloud_cover': 0.0, 'proj:epsg': 32643, 'proj:bbox': [600000.0, 3090240.0, 709800.0, 3200040.0], 'proj:geometry': {'type': 'MultiPolygon', 'crs': {'type': 'name', 'properties': {'name': 'urn:ogc:def:crs:EPSG::32643'}}, 'coordinates': [[[[600001.0000001469, 3200038.9999529934], [709799.000025516, 3200038.999952611], [709799.0000228899, 3090240.9999508364], [600001.0000001314, 3090240.999951191], [600001.0000001469, 3200038.9999529934]]]]}}


In [27]:
# Convert a raw Sentinel Hub catalog item into our application's scene format.

def normalize_scene(feature: dict) -> dict:

    properties = feature.get("properties", {})

    cloud_cover = properties.get("eo:cloud_cover")

    if cloud_cover is not None:
        cloud_cover = float(cloud_cover)

    return {
        "scene_id": feature.get("id"),
        "collection": "sentinel-2-l2a",
        "acquisition_time": properties.get("datetime"),
        "cloud_cover": cloud_cover,
        "bbox": feature.get("bbox"),
        "geometry": feature.get("geometry"),
    }

In [28]:
# Normalize all catalog results into the internal scene representation.

normalized_scenes = [
    normalize_scene(scene)
    for scene in scenes
]

print("Normalized scenes:", len(normalized_scenes))

Normalized scenes: 5


In [29]:
# Display the main metadata for each candidate scene.

for index, scene in enumerate(normalized_scenes, start=1):

    print(f"Scene {index}")
    print("ID:", scene["scene_id"])
    print("Acquisition:", scene["acquisition_time"])
    print("Cloud cover:", scene["cloud_cover"])
    print("BBox:", scene["bbox"])
    print("-" * 70)

Scene 1
ID: S2C_MSIL2A_20250128T053131_N0511_R105_T43RFM_20250128T084454
Acquisition: 2025-01-28T05:41:34Z
Cloud cover: 0.0
BBox: [76.01641680775477, 27.92065311807928, 77.15188453201881, 28.92436500128604]
----------------------------------------------------------------------
Scene 2
ID: S2C_MSIL2A_20250128T053131_N0511_R105_T43RGM_20250128T084454
Acquisition: 2025-01-28T05:41:31Z
Cloud cover: 0.0
BBox: [77.03202594439398, 27.901201701204435, 78.17607277525508, 28.912650957754558]
----------------------------------------------------------------------
Scene 3
ID: S2B_MSIL2A_20250123T053029_N0511_R105_T43RFM_20250123T084836
Acquisition: 2025-01-23T05:41:08Z
Cloud cover: 0.02
BBox: [76.01641680775477, 27.92065311807928, 77.15188453201881, 28.92436500128604]
----------------------------------------------------------------------
Scene 4
ID: S2B_MSIL2A_20250123T053029_N0511_R105_T43RGM_20250123T084836
Acquisition: 2025-01-23T05:41:05Z
Cloud cover: 8.9
BBox: [77.03202594439398, 27.9012017012

In [30]:
# Select the most suitable Sentinel-2 scene from normalized catalog results.

def select_best_scene(
    scenes: list[dict],
) -> dict | None:

    valid_scenes = [
        scene
        for scene in scenes
        if scene.get("scene_id")
        and scene.get("cloud_cover") is not None
        and scene.get("acquisition_time")
    ]

    if not valid_scenes:
        return None

    return min(
        valid_scenes,
        key=lambda scene: (
            scene["cloud_cover"],
            scene["acquisition_time"],
        ),
    )

In [31]:
# Select the best available scene.

best_scene = select_best_scene(normalized_scenes)

if best_scene:
    print("Best scene selected:")
    print(best_scene)
else:
    print("No suitable Sentinel-2 scene was found.")

Best scene selected:
{'scene_id': 'S2C_MSIL2A_20250128T053131_N0511_R105_T43RGM_20250128T084454', 'collection': 'sentinel-2-l2a', 'acquisition_time': '2025-01-28T05:41:31Z', 'cloud_cover': 0.0, 'bbox': [77.03202594439398, 27.901201701204435, 78.17607277525508, 28.912650957754558], 'geometry': {'type': 'MultiPolygon', 'crs': {'type': 'name', 'properties': {'name': 'urn:ogc:def:crs:OGC::CRS84'}}, 'coordinates': [[[[77.05103194254626, 28.912650957754558], [78.17607277525508, 28.890809106824655], [78.1466761858211, 27.901201701204435], [77.03202594439398, 27.92216395441563], [77.05103194254626, 28.912650957754558]]]]}}


In [32]:
# Build the Sentinel-2 Evalscript used by the Processing API.

def build_evalscript(request: SatelliteRequest) -> str:

    if request.modality == "optical":

        bands = ["B04", "B03", "B02"]

        return """
//VERSION=3

function setup() {
    return {
        input: [{
            bands: ["B04", "B03", "B02"]
        }],
        output: {
            bands: 3,
            sampleType: "UINT16"
        }
    };
}

function evaluatePixel(sample) {
    return [
        sample.B04 * 10000,
        sample.B03 * 10000,
        sample.B02 * 10000
    ];
}
"""

    if request.modality == "multispectral":

        if not request.bands:
            raise ValueError(
                "bands must be provided when modality='multispectral'."
            )

        bands = request.bands

        band_list = ", ".join(
            f'"{band}"'
            for band in bands
        )

        sample_values = ", ".join(
            f"sample.{band}"
            for band in bands
        )

        return f"""
//VERSION=3

function setup() {{
    return {{
        input: [{{
            bands: [{band_list}]
        }}],
        output: {{
            bands: {len(bands)},
            sampleType: "FLOAT32"
        }}
    }};
}}

function evaluatePixel(sample) {{
    return [{sample_values}];
}}
"""

    raise ValueError(
        f"Unsupported modality: {request.modality}"
    )

In [33]:
# Build the Processing API request for the selected Sentinel-2 acquisition.

def build_process_payload(
    request: SatelliteRequest,
    scene: dict,
) -> dict:

    evalscript = build_evalscript(request)

    acquisition_time = scene.get("acquisition_time")

    if not acquisition_time:
        raise ValueError(
            "Selected scene does not contain an acquisition time."
        )

    # Use the selected scene's acquisition instant rather than
    # re-searching the entire user-provided date range.
    acquisition_dt = datetime.fromisoformat(
        acquisition_time.replace("Z", "+00:00")
    )

    payload = {
        "input": {
            "bounds": {
                "bbox": request.bbox,
                "properties": {
                    "crs": (
                        "http://www.opengis.net/def/crs/"
                        "OGC/1.3/CRS84"
                    )
                }
            },

            "data": [
                {
                    "type": "sentinel-2-l2a",
                    "dataFilter": {
                        "timeRange": {
                            "from": acquisition_dt.isoformat(),
                            "to": acquisition_dt.isoformat(),
                        },
                        "mosaickingOrder": "leastCC",
                        "maxCloudCoverage": request.max_cloud_cover,
                    }
                }
            ]
        },

        "output": {
            "width": request.width,
            "height": request.height,
            "responses": [
                {
                    "identifier": "default",
                    "format": {
                        "type": "image/tiff"
                    }
                }
            ]
        },

        "evalscript": evalscript,
    }

    return payload

In [34]:
# Create a unique, readable output filename for each imagery request.

def create_output_path(
    request: SatelliteRequest,
    scene: dict | None = None,
) -> Path:

    timestamp = datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%SZ"
    )

    scene_id = "unknown_scene"

    if scene and scene.get("scene_id"):
        scene_id = scene["scene_id"]

    safe_scene_id = re.sub(
        r"[^A-Za-z0-9_-]",
        "_",
        scene_id,
    )

    filename = (
        f"sentinel2_{safe_scene_id}_"
        f"{request.start_date}_{request.end_date}_"
        f"{request.modality}_{timestamp}.tif"
    )

    return CONFIG.output_dir / filename

In [35]:
# Execute the Processing API request for the selected scene and save the GeoTIFF.

def process_satellite_imagery(
    request: SatelliteRequest,
    scene: dict,
) -> Path:

    payload = build_process_payload(
        request,
        scene,
    )

    output_path = create_output_path(
        request,
        scene,
    )

    response = http_client.request(
        "POST",
        CONFIG.process_url,
        json=payload,
        content_type="application/json",
    )

    if not response.ok:

        try:
            error_data = response.json()

        except ValueError:
            error_data = response.text

        raise RuntimeError(
            f"Processing API failed "
            f"(HTTP {response.status_code}): "
            f"{error_data}"
        )

    content_type = response.headers.get(
        "Content-Type",
        ""
    ).lower()

    if "image/tiff" not in content_type:

        raise RuntimeError(
            "Processing API returned an unexpected "
            f"content type: {content_type}"
        )

    output_path.write_bytes(
        response.content
    )

    if output_path.stat().st_size == 0:
        output_path.unlink(missing_ok=True)

        raise RuntimeError(
            "Processing API returned an empty GeoTIFF."
        )

    return output_path

In [36]:
# Validate the downloaded GeoTIFF and extract its core raster metadata.

def validate_raster(
    raster_path: Path,
) -> dict:

    if not raster_path.exists():
        raise RuntimeError(
            f"Raster file does not exist: {raster_path}"
        )

    if raster_path.stat().st_size == 0:
        raise RuntimeError(
            "Raster file is empty."
        )

    try:
        with rasterio.open(raster_path) as src:

            if src.width <= 0 or src.height <= 0:
                raise RuntimeError(
                    "Raster has invalid dimensions."
                )

            if src.count <= 0:
                raise RuntimeError(
                    "Raster contains no bands."
                )

            if src.crs is None:
                raise RuntimeError(
                    "Raster does not contain a CRS."
                )

            if src.bounds is None:
                raise RuntimeError(
                    "Raster does not contain spatial bounds."
                )

            return {
                "width": src.width,
                "height": src.height,
                "band_count": src.count,
                "dtype": src.dtypes[0],
                "crs": str(src.crs),
                "bounds": {
                    "left": src.bounds.left,
                    "bottom": src.bounds.bottom,
                    "right": src.bounds.right,
                    "top": src.bounds.top,
                },
                "resolution": {
                    "x": src.res[0],
                    "y": src.res[1],
                },
                "nodata": src.nodata,
            }

    except rasterio.errors.RasterioIOError as e:
        raise RuntimeError(
            f"Downloaded file is not a valid readable GeoTIFF: {e}"
        )

In [37]:
# Build the normalized result returned by the satellite-data tool.

def build_success_result(
    request: SatelliteRequest,
    scene: dict | None,
    raster_path: Path,
    raster_metadata: dict,
) -> dict:

    cloud_cover = None

    if scene:
        cloud_cover = scene.get("cloud_cover")

    optical_quality_poor = (
        cloud_cover is not None
        and cloud_cover > 50
    )

    return {
        "status": "success",

        "data": {
            "file_path": str(raster_path),
            "file_name": raster_path.name,
            "format": request.output_format,
        },

        "source": {
            "provider": "Sentinel Hub",
            "collection": "sentinel-2-l2a",
            "scene_id": (
                scene.get("scene_id")
                if scene
                else None
            ),
            "acquisition_time": (
                scene.get("acquisition_time")
                if scene
                else None
            ),
        },

        "request": {
            "bbox": request.bbox,
            "start_date": request.start_date,
            "end_date": request.end_date,
            "modality": request.modality,
            "bands": request.bands,
            "max_cloud_cover": request.max_cloud_cover,
        },

        "raster": raster_metadata,

        "quality": {
            "cloud_cover": cloud_cover,
            "valid": True,
        },

        "flags": {
            "optical_quality_poor": optical_quality_poor,
            "sar_recommended": optical_quality_poor,
        },
    }

In [38]:
# Main satellite imagery retrieval function used by the final tool.

def fetch_satellite_imagery(
    request: SatelliteRequest,
) -> dict:

    # 1. Search for available Sentinel-2 observations.
    raw_scenes = search_sentinel2_catalog(
        request,
        max_results=20,
    )

    # 2. Normalize catalog results.
    scenes = [
        normalize_scene(scene)
        for scene in raw_scenes
    ]

    # 3. Select the best available scene.
    best_scene = select_best_scene(scenes)

    # 4. Stop cleanly when no suitable imagery exists.
    if best_scene is None:

        return {
            "status": "error",
            "error": {
                "type": "no_suitable_scene",
                "message": (
                    "No suitable Sentinel-2 L2A "
                    "scene was found for the requested "
                    "AOI, time range and cloud threshold."
                ),
            },
        }

    # 5. Request the actual imagery.
    raster_path = process_satellite_imagery(
        request,
        scene=best_scene,
    )

    # 6. Validate the downloaded raster.
    raster_metadata = validate_raster(
        raster_path
    )

    # 7. Return the normalized final result.
    return build_success_result(
        request=request,
        scene=best_scene,
        raster_path=raster_path,
        raster_metadata=raster_metadata,
    )

In [39]:
# Execute the complete satellite-data retrieval pipeline.

try:

    final_request = SatelliteRequest(
        bbox=[77.10, 28.50, 77.30, 28.70],
        start_date="2025-01-01",
        end_date="2025-01-31",
        modality="optical",
        bands=None,
        max_cloud_cover=30,
        width=512,
        height=512,
        crs="EPSG:4326",
        output_format="GeoTIFF",
    )

    result = fetch_satellite_imagery(
        final_request
    )

    print("Tool execution completed.")
    print("Status:", result["status"])

    if result["status"] == "success":

        print("\nImage:")
        print(result["data"]["file_path"])

        print("\nScene:")
        print(result["source"]["scene_id"])

        print("\nAcquisition:")
        print(result["source"]["acquisition_time"])

        print("\nCloud cover:")
        print(result["quality"]["cloud_cover"])

        print("\nRaster:")
        print(result["raster"])

    else:

        print("\nError:")
        print(result["error"])

except Exception as e:

    print("Tool execution failed.")
    print(e)

Tool execution completed.
Status: success

Image:
sih_satellite_data/sentinel2_S2C_MSIL2A_20250128T053131_N0511_R105_T43RGM_20250128T084454_2025-01-01_2025-01-31_optical_20260902T180322Z.tif

Scene:
S2C_MSIL2A_20250128T053131_N0511_R105_T43RGM_20250128T084454

Acquisition:
2025-01-28T05:41:31Z

Cloud cover:
0.0

Raster:
{'width': 512, 'height': 512, 'band_count': 3, 'dtype': 'uint16', 'crs': 'EPSG:4326', 'bounds': {'left': 77.1, 'bottom': 28.5, 'right': 77.3, 'top': 28.7}, 'resolution': {'x': 0.00039062500000000555, 'y': 0.0003906249999999986}, 'nodata': None}


In [40]:
# Install FastMCP for exposing the satellite service through MCP.

!pip -q install fastmcp

In [41]:
# Import FastMCP for creating the MCP server.

from fastmcp import FastMCP

In [42]:
# Create the MCP server that will expose the satellite imagery tool.

mcp = FastMCP(
    name="SIH Satellite Data Server"
)

print("MCP server initialized.")

MCP server initialized.


In [43]:
# Expose Sentinel-2 satellite imagery retrieval as an MCP tool.

@mcp.tool(
    name="fetch_satellite_imagery",
    description=(
        "Fetch Sentinel-2 L2A satellite imagery for a requested "
        "bounding box and date range. Supports optical RGB or "
        "multispectral output in GeoTIFF format."
    ),
)
def mcp_fetch_satellite_imagery(
    bbox: list[float],
    start_date: str,
    end_date: str,
    modality: str = "optical",
    bands: list[str] | None = None,
    max_cloud_cover: float = 30.0,
    width: int = 512,
    height: int = 512,
    crs: Literal["EPSG:4326"] = "EPSG:4326",
) -> dict:
    """
    Retrieve Sentinel-2 L2A imagery.

    bbox:
        [min_lon, min_lat, max_lon, max_lat]

    start_date / end_date:
        Date range in YYYY-MM-DD format.

    modality:
        "optical" or "multispectral".

    bands:
        Required for multispectral mode.
        Omit for optical RGB mode.

    max_cloud_cover:
        Maximum allowed cloud-cover percentage.

    width / height:
        Requested output dimensions.

    crs:
        Coordinate reference system for the request.
    """

    try:

        request = SatelliteRequest(
            bbox=bbox,
            start_date=start_date,
            end_date=end_date,
            modality=modality,
            bands=bands,
            max_cloud_cover=max_cloud_cover,
            width=width,
            height=height,
            crs=crs,
            output_format="GeoTIFF",
        )

        return fetch_satellite_imagery(
            request
        )

    except ValueError as e:

        return {
            "status": "error",
            "error": {
                "type": "validation_error",
                "message": str(e),
            },
        }

    except RuntimeError as e:

        return {
            "status": "error",
            "error": {
                "type": "service_error",
                "message": str(e),
            },
        }

    except Exception as e:

        return {
            "status": "error",
            "error": {
                "type": "unexpected_error",
                "message": str(e),
            },
        }